In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from Bio import SeqIO

In [2]:
dataset_home = "~/Desktop/datasets"
datatasets = {
    "hiv": f"{dataset_home}/original/hiv/data_clean",
    "sars_kevolve": f"{dataset_home}/original/sars_cov2/data_clean",
    "denv_gramep": f"{dataset_home}/original/denv/data_clean",
    "hbv": f"{dataset_home}/original/HBV/data_clean",
    "mkpx": f"{dataset_home}/original/mkpx/data_clean",
    "sars_ncbi": f"{dataset_home}/new_sars_denv/denv_sars_cov_2/sars_ncbi_clean/test",
    "denv_ncbi": f"{dataset_home}/new_sars_denv/denv_sars_cov_2/denv_ncbi_clean/test",
}

In [3]:
rows = []
for name, path in datatasets.items():
    root = Path(path).expanduser()
    subs = [d for d in root.iterdir() if d.is_dir()]
    # datasets organised by class folder: ignore the train/test split copies;
    # the NCBI ones only have train/test, so there we merge both back together
    classdirs = [d for d in subs if d.name not in ("train", "test")] or subs

    by_class = defaultdict(list)
    for d in classdirs:
        for fasta in d.rglob("*.fasta"):
            by_class[fasta.stem] += [len(r.seq) for r in SeqIO.parse(fasta, "fasta")]

    for cls, seqs_len in sorted(by_class.items()):
        mean = np.mean(seqs_len)
        rows.append({
            "dataset": name,
            "class": cls,
            "n_seqs": len(seqs_len),
            "mean": mean,
            "sd": np.std(seqs_len),
            "variation": np.std(seqs_len) / mean,
            "min": min(seqs_len),
            "max": max(seqs_len),
        })

df = pd.DataFrame(rows)
df

,dataset,class,n_seqs,mean,sd,variation,min,max
0,hiv,HIV1_A,133,8878.300752,438.020976,0.049336,8009,9854
1,hiv,HIV1_B,128,9351.789062,441.392287,0.047199,8005,9915
2,hiv,HIV1_C,130,9148.084615,349.096108,0.038161,8134,9913
3,hiv,HIV1_D,86,8963.069767,386.529002,0.043125,8143,9839
4,hiv,HIV1_F,90,8944.622222,353.483131,0.039519,8096,9802
5,hiv,HIV1_G,94,9041.106383,363.940651,0.040254,8003,9792
6,sars_kevolve,Alpha,139506,29835.111544,55.618752,0.001864,29419,29904
7,sars_kevolve,Beta,647,29811.004637,57.532156,0.001930,29636,29903
8,sars_kevolve,Delta,8424,29801.185066,63.068428,0.002116,29402,29912
9,sars_kevolve,Epsilon,13649,29834.784013,62.382077,0.002091,29430,29912


In [4]:
rows = []
for name, path in datatasets.items():
    root = Path(path).expanduser()
    for fasta in sorted(root.rglob("*.fasta")):
        seqs_len = [len(r.seq) for r in SeqIO.parse(fasta, "fasta")]
        if not seqs_len:
            continue
        mean = np.mean(seqs_len)
        rows.append({
            "dataset": name,
            "file": str(fasta.relative_to(root)),
            "n_seqs": len(seqs_len),
            "mean": mean,
            "sd": np.std(seqs_len),
            "variation": np.std(seqs_len) / mean,
            "min": min(seqs_len),
            "max": max(seqs_len),
        })

df = pd.DataFrame(rows)
df

,dataset,file,n_seqs,mean,sd,variation,min,max
0,hiv,HIV1_A/HIV1_A.fasta,133,8878.300752,438.020976,0.049336,8009,9854
1,hiv,HIV1_B/HIV1_B.fasta,128,9351.789062,441.392287,0.047199,8005,9915
2,hiv,HIV1_C/HIV1_C.fasta,130,9148.084615,349.096108,0.038161,8134,9913
3,hiv,HIV1_D/HIV1_D.fasta,86,8963.069767,386.529002,0.043125,8143,9839
4,hiv,HIV1_F/HIV1_F.fasta,90,8944.622222,353.483131,0.039519,8096,9802
...,...,...,...,...,...,...,...,...
93,sars_ncbi,Omicron.fasta,1132,29771.563604,44.423719,0.001492,29540,30128
94,denv_ncbi,type1.fasta,4638,10514.736956,209.324615,0.019908,10168,11195
95,denv_ncbi,type2.fasta,4235,10512.683117,203.076261,0.019317,10168,10940
96,denv_ncbi,type3.fasta,2356,10512.208829,202.379094,0.019252,10168,10810


## Redução do GREAC

Para cada FASTA que tenha ao lado a versão reduzida gerada pelo `fasta-regions`
(`DataIO.createFastaRegionFile`, que escreve `<classe>-extracted.fasta`), compara o
tamanho em disco do ficheiro bruto antes e depois.

In [6]:
rows = []
for name, path in datatasets.items():
    root = Path(path).expanduser()
    # o código escreve <classe>-extracted.fasta; aceita tambem _extracted
    for red in sorted(set(root.rglob("*-extracted.fasta")) | set(root.rglob("*_extracted.fasta"))):
        original = red.parent / f"{red.stem.rsplit('-extracted', 1)[0].rsplit('_extracted', 1)[0]}.fasta"
        if not original.exists():
            print(f"sem original para {red}")
            continue
        before, after = original.stat().st_size, red.stat().st_size
        rows.append({
            "dataset": name,
            "class": original.stem,
            "bytes_before": before,
            "bytes_after": after,
            "mb_before": before / 1024**2,
            "mb_after": after / 1024**2,
            "retained_pct": 100 * after / before,
            "reduction_pct": 100 * (1 - after / before),
        })

df_reduction = pd.DataFrame(rows)
df_reduction

,dataset,class,bytes_before,bytes_after,mb_before,mb_after,retained_pct,reduction_pct
0,hiv,HIV1_A,1202821,837578,1.147099,0.798777,69.634468,30.365532
1,hiv,HIV1_B,1219219,806065,1.162738,0.768723,66.113225,33.886775
2,hiv,HIV1_C,1211346,818740,1.155230,0.780811,67.589277,32.410723
3,hiv,HIV1_D,785167,541618,0.748794,0.516527,68.981249,31.018751
4,hiv,HIV1_F,820002,566793,0.782015,0.540536,69.120929,30.879071
5,hiv,HIV1_G,865674,591962,0.825571,0.564539,68.381631,31.618369
6,sars_kevolve,Alpha,4235679101,649657384,4039.458371,619.561562,15.337738,84.662262
7,sars_kevolve,Beta,19627624,2998480,18.718361,2.859573,15.276836,84.723164
8,sars_kevolve,Delta,255478261,39012731,243.643056,37.205440,15.270470,84.729530
9,sars_kevolve,Epsilon,414432279,63850682,395.233420,60.892756,15.406783,84.593217


In [7]:
df_reduction_total = (
    df_reduction.groupby("dataset")[["bytes_before", "bytes_after"]].sum()
    .assign(
        mb_before=lambda d: d.bytes_before / 1024**2,
        mb_after=lambda d: d.bytes_after / 1024**2,
        retained_pct=lambda d: 100 * d.bytes_after / d.bytes_before,
        reduction_pct=lambda d: 100 * (1 - d.bytes_after / d.bytes_before),
    )
    .reset_index()
)
df_reduction_total

,dataset,bytes_before,bytes_after,mb_before,mb_after,retained_pct,reduction_pct
0,denv_gramep,62122286,48046548,59.244429,45.820759,77.341887,22.658113
1,hbv,21209469,13489352,20.226926,12.864449,63.600612,36.399388
2,hiv,6104229,4162756,5.821446,3.969913,68.194624,31.805376
3,mkpx,680469776,429729747,648.946548,409.822223,63.151923,36.848077
4,sars_kevolve,8396983425,1283291047,8007.987428,1223.841712,15.282763,84.717237
